# Training-Time Ensemble — 12-Feature Cache

This notebook loads **pre-computed `.npy` probability files** from a saved cache folder.
No base models are loaded or run.

This version uses the **12-feature cache** (3 probabilities per model) and keeps the
original notebook-family MLP (`StackingMLP`) only.

## Cache folder layout expected
```
stacking_cache_train_split_12feat/
  snli_tr_1_1__train__bert_probs.npy
  snli_tr_1_1__train__mdeberta_probs.npy
  ...  (one file per config × split × model)
```


In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

from datasets import load_dataset
from IPython.display import display

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

MODEL_ORDER  = ["bert", "mdeberta", "gemma", "qwen"]
DATASET_NAME = "yilmazzey/sdp2-nli"
CONFIGS      = ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]
LABEL_MAP    = {0: "entailment", 1: "neutral", 2: "contradiction"}
LABEL_NAMES  = [LABEL_MAP[i] for i in range(3)]
NUM_LABELS   = 3

EVAL_SPLITS = {
    "snli_tr_1_1": ["test"],
    "multinli_tr_1_1": ["validation_matched", "validation_mismatched"],
    "trglue_mnli": ["test_matched", "test_mismatched"],
}

STATIC_WEIGHTED_WEIGHTS = {"bert": 0.25, "mdeberta": 0.12, "gemma": 0.26, "qwen": 0.37}
ROUTER_W      = {"bert": 0.0, "mdeberta": 0.0, "gemma": 0.5, "qwen": 0.5}
CLASS_WEIGHTS_V1 = {
    0: {"bert": 0.0, "mdeberta": 0.05, "gemma": 0.25, "qwen": 0.70},
    1: {"bert": 0.0, "mdeberta": 0.10, "gemma": 0.70, "qwen": 0.20},
    2: {"bert": 0.0, "mdeberta": 0.05, "gemma": 0.25, "qwen": 0.70},
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CACHE_DIR = Path("stacking_cache_train_split_12feat")
ARTIFACT_DIR = Path("stacking_artifacts_12feat")
ARTIFACT_DIR.mkdir(exist_ok=True)

assert CACHE_DIR.exists(), (
    f"Cache folder not found: {CACHE_DIR.resolve()}\n"
    "Set CACHE_DIR to the folder that contains the .npy probability files."
)
print("Cache folder:", CACHE_DIR.resolve())

META_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 8
META_BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

FEATURE_NAMES = [
    "bert_p0", "bert_p1", "bert_p2",
    "mdeberta_p0", "mdeberta_p1", "mdeberta_p2",
    "gemma_p0", "gemma_p1", "gemma_p2",
    "qwen_p0", "qwen_p1", "qwen_p2",
]


## Load labels from HuggingFace (no model inference)

In [ ]:
# Read HF token from environment — never hard-code tokens.
hf_token = os.environ.get("HF_TOKEN", None)
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("HF_TOKEN not set — assuming public dataset or already logged in.")

datasets_hf = {}
for cfg in CONFIGS:
    print(f"Loading labels: {DATASET_NAME} :: {cfg}")
    datasets_hf[cfg] = load_dataset(DATASET_NAME, cfg)
    print("  splits:", list(datasets_hf[cfg].keys()))
print("Label loading complete.")


In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

## Cache loading utilities

In [ ]:
def cache_name(config: str, split: str, model_key: str) -> Path:
    safe_cfg = config.replace("/", "_")
    safe_sp  = split.replace("/", "_")
    return CACHE_DIR / f"{safe_cfg}__{safe_sp}__{model_key}_probs.npy"


def load_probs_from_cache(config: str, split: str) -> dict:
    """Load all four model probability arrays from .npy files. No model inference."""
    probs = {}
    for model_key in MODEL_ORDER:
        p = cache_name(config, split, model_key)
        if not p.exists():
            raise FileNotFoundError(
                f"Missing cache file: {p}\n"
                f"Run the feature-extraction notebook first."
            )
        probs[model_key] = np.load(p)
    shapes = {k: v.shape for k, v in probs.items()}
    print(f"  Loaded {config}/{split} — shapes: {shapes}")
    return probs


## Feature engineering

In [ ]:
def build_X12(probs_dict: dict) -> np.ndarray:
    """
    12 features per example, column layout:
      [0:3]  bert_p0  bert_p1  bert_p2
      [3:6]  mdeberta_p0  mdeberta_p1  mdeberta_p2
      [6:9]  gemma_p0  gemma_p1  gemma_p2
      [9:12] qwen_p0  qwen_p1  qwen_p2
    """
    return np.hstack([
        probs_dict["bert"],
        probs_dict["mdeberta"],
        probs_dict["gemma"],
        probs_dict["qwen"],
    ]).astype(np.float32)


## Static ensemble helpers

In [ ]:
def onehot_from_probs(p: np.ndarray) -> np.ndarray:
    out = np.zeros_like(p)
    out[np.arange(len(p)), p.argmax(axis=1)] = 1.0
    return out


def weighted_static_pred(bert_p, mdeb_p, gemma_p, qwen_p):
    w     = np.array([STATIC_WEIGHTED_WEIGHTS[m] for m in MODEL_ORDER], dtype=np.float32)
    probs = np.stack([bert_p, mdeb_p, gemma_p, qwen_p], axis=0)
    return np.einsum("m,mnc->nc", w, probs).argmax(axis=1)


def majority_vote_pred(bert_p, mdeb_p, gemma_p, qwen_p):
    votes = np.stack(
        [bert_p.argmax(1), mdeb_p.argmax(1), gemma_p.argmax(1), qwen_p.argmax(1)], axis=1
    )
    return np.array(
        [int(Counter(row.tolist()).most_common(1)[0][0]) for row in votes], dtype=np.int64
    )


def class_conditional_routing_pred(bert_p, mdeb_p, gemma_p, qwen_p):
    oh = np.stack([
        onehot_from_probs(bert_p), onehot_from_probs(mdeb_p),
        onehot_from_probs(gemma_p), onehot_from_probs(qwen_p),
    ], axis=0)
    router_w    = np.array([ROUTER_W[m] for m in MODEL_ORDER], dtype=np.float32)
    routed_class = np.einsum("m,mnc->nc", router_w, oh).argmax(axis=1)
    final = np.empty(len(routed_class), dtype=np.int64)
    for c in range(NUM_LABELS):
        mask = routed_class == c
        if not mask.any():
            continue
        w_c = np.array([CLASS_WEIGHTS_V1[c][m] for m in MODEL_ORDER], dtype=np.float32)
        final[mask] = np.einsum("m,mnc->nc", w_c, oh[:, mask, :]).argmax(axis=1)
    return final


## Meta-learner architectures

In [ ]:
class StackingMLP(nn.Module):
    """Batchnorm MLP: 12 -> 64 -> 32 -> 3."""
    def __init__(self, in_dim=12, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)


class StackingBiLSTM(nn.Module):
    """Grouped BiLSTM over per-model 3-probability vectors."""
    def __init__(self, hidden_size=64, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=3, hidden_size=hidden_size,
            num_layers=1, batch_first=True, bidirectional=True,
        )
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size * 2, n_classes)

    def forward(self, x):
        seq = x.view(x.size(0), 4, 3)
        out, _ = self.lstm(seq)
        return self.fc(self.drop(out[:, -1, :]))


class StackingBiLSTM_dynamic(nn.Module):
    """Variable-model-count grouped BiLSTM for ablations."""
    def __init__(self, num_models=4, hidden_size=64, n_classes=3):
        super().__init__()
        self.num_models = num_models
        self.lstm = nn.LSTM(
            input_size=3, hidden_size=hidden_size,
            num_layers=1, batch_first=True, bidirectional=True,
        )
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size * 2, n_classes)

    def forward(self, x):
        seq = x.view(x.size(0), self.num_models, 3)
        out, _ = self.lstm(seq)
        return self.fc(self.drop(out[:, -1, :]))


def build_Xreduced(probs_dict: dict, model_keys: list) -> np.ndarray:
    """Build reduced matrix in [p0, p1, p2] blocks for selected models."""
    return np.hstack([probs_dict[k] for k in model_keys]).astype(np.float32)


class StackingLSTM_flat(nn.Module):
    """Ungrouped / flat BiLSTM over 12 scalar timesteps."""
    def __init__(self, hidden_size=64, n_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1, hidden_size=hidden_size,
            num_layers=1, batch_first=True, bidirectional=True,
        )
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size * 2, n_classes)

    def forward(self, x):
        seq = x.unsqueeze(-1)
        out, _ = self.lstm(seq)
        return self.fc(self.drop(out[:, -1, :]))


## Training loop

In [ ]:
def _train_torch_model(model, X_train, y_train, model_name="model"):
    X_t = torch.tensor(X_train, dtype=torch.float32)
    y_t = torch.tensor(y_train, dtype=torch.long)

    n = len(X_t)
    n_val = max(1, int(0.10 * n))
    train_ds, val_ds = random_split(
        TensorDataset(X_t, y_t), [n - n_val, n_val],
        generator=torch.Generator().manual_seed(SEED),
    )
    train_loader = DataLoader(train_ds, batch_size=META_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=META_BATCH_SIZE, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=META_EPOCHS, eta_min=1e-5)

    best_state, best_val, bad_epochs = None, float("inf"), 0
    model = model.to(DEVICE)

    for epoch in range(META_EPOCHS):
        model.train()
        tr_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tr_losses.append(loss.item())
        scheduler.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_losses.append(criterion(model(xb), yb).item())

        mean_tr = float(np.mean(tr_losses)) if tr_losses else float("nan")
        mean_val = float(np.mean(val_losses)) if val_losses else float("nan")
        print(f"{model_name} epoch {epoch+1:02d}/{META_EPOCHS} | train_loss={mean_tr:.4f}  val_loss={mean_val:.4f}")

        if mean_val < best_val - 1e-5:
            best_val = mean_val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping {model_name} at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def _get_accuracy_batched(model, X_data, y_true, batch_size):
    model.eval()
    preds = []
    y_true_list = []
    dataset = TensorDataset(torch.tensor(X_data, dtype=torch.float32), torch.tensor(y_true, dtype=torch.long))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(DEVICE)
            outputs = model(xb)
            preds.append(outputs.argmax(-1).cpu().numpy())
            y_true_list.append(yb.cpu().numpy())
    return accuracy_score(np.concatenate(y_true_list), np.concatenate(preds))


def train_meta_learners(X_train: np.ndarray, y_train: np.ndarray):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train).astype(np.float32)

    mlp = StackingMLP(in_dim=12, n_classes=NUM_LABELS)
    mlp = _train_torch_model(mlp, Xs, y_train, model_name="MLP")

    svm = LinearSVC(C=1.0, class_weight="balanced", random_state=SEED, max_iter=20000)
    svm.fit(Xs, y_train)
    print("LinearSVC fitted.")

    bilstm = StackingBiLSTM(hidden_size=64, n_classes=NUM_LABELS)
    bilstm = _train_torch_model(bilstm, Xs, y_train, model_name="BiLSTM")

    flat_lstm = StackingLSTM_flat(hidden_size=64, n_classes=NUM_LABELS)
    flat_lstm = _train_torch_model(flat_lstm, Xs, y_train, model_name="FlatLSTM")

    mlp_acc = _get_accuracy_batched(mlp, Xs, y_train, META_BATCH_SIZE)
    bilstm_acc = _get_accuracy_batched(bilstm, Xs, y_train, META_BATCH_SIZE)
    flat_lstm_acc = _get_accuracy_batched(flat_lstm, Xs, y_train, META_BATCH_SIZE)

    print(
        f"Train acc | MLP: {mlp_acc:.4f} "
        f"| SVC: {(svm.predict(Xs)==y_train).mean():.4f} | BiLSTM: {bilstm_acc:.4f} | FlatLSTM: {flat_lstm_acc:.4f}"
    )

    return {
        "scaler": scaler,
        "mlp": mlp,
        "svm": svm,
        "bilstm": bilstm,
        "flat_lstm": flat_lstm,
    }


## Metrics and display helpers

In [ ]:
def compute_metrics_dict(y_true, y_pred):
    acc    = float(accuracy_score(y_true, y_pred))
    f1m    = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_each = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per  = {LABEL_NAMES[i]: float(f1_each[i]) for i in range(NUM_LABELS)}
    cm      = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return {"accuracy": acc, "f1_macro": f1m, "f1_per_class": f1_per, "cm": cm}


def plot_confusion(cm, title):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout(); plt.show()


def append_row(rows, split_key, method, y_true, y_pred, row_kind="computed"):
    m = compute_metrics_dict(y_true, y_pred)
    rows.append({
        "split": split_key, "method": method, "row_kind": row_kind,
        "accuracy": m["accuracy"], "f1_macro": m["f1_macro"],
        "f1_entailment": m["f1_per_class"]["entailment"],
        "f1_neutral":    m["f1_per_class"]["neutral"],
        "f1_contradiction": m["f1_per_class"]["contradiction"],
    })


def append_reference_row(rows, split_key):
    # Reference accuracy only exists for trglue_mnli::test_matched
    ref_acc = 0.8408 if split_key == "trglue_mnli::test_matched" else float("nan")
    rows.append({
        "split": split_key,
        "method": "REF_published_static_weighted_84.08pct",
        "row_kind": "reference_published",
        "accuracy": ref_acc,
        "f1_macro": float("nan"), "f1_entailment": float("nan"),
        "f1_neutral": float("nan"), "f1_contradiction": float("nan"),
    })


## Main training + evaluation loop

In [ ]:
rows = []
trained_meta = {}
train_feature_bank = {}
eval_feature_bank = {}

for cfg in CONFIGS:
    print("\n" + "="*100)
    print(f"Config: {cfg}")

    y_train = np.array(datasets_hf[cfg]["train"]["label"], dtype=np.int64)
    train_probs = load_probs_from_cache(cfg, "train")
    X_train = build_X12(train_probs)
    print(f"  Train features shape: {X_train.shape}")

    meta = train_meta_learners(X_train, y_train)
    trained_meta[cfg] = meta
    train_feature_bank[cfg] = {"X_train": X_train, "y_train": y_train, "probs": train_probs}
    eval_feature_bank[cfg] = {}

    for sp in EVAL_SPLITS[cfg]:
        split_key = f"{cfg}::{sp}"
        y_true = np.array(datasets_hf[cfg][sp]["label"], dtype=np.int64)
        eval_probs = load_probs_from_cache(cfg, sp)
        X_eval = build_X12(eval_probs)
        X_eval_s = meta["scaler"].transform(X_eval).astype(np.float32)

        eval_feature_bank[cfg][sp] = {"X_eval_s": X_eval_s, "y_true": y_true, "probs": eval_probs}

        Xt = torch.tensor(X_eval_s, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            pred_mlp = meta["mlp"](Xt).argmax(-1).cpu().numpy()
            pred_bilstm = meta["bilstm"](Xt).argmax(-1).cpu().numpy()
            pred_flat_lstm = meta["flat_lstm"](Xt).argmax(-1).cpu().numpy()
        pred_svm = meta["svm"].predict(X_eval_s)

        pred_bert = eval_probs["bert"].argmax(axis=1)
        pred_mdeb = eval_probs["mdeberta"].argmax(axis=1)
        pred_gemma = eval_probs["gemma"].argmax(axis=1)
        pred_qwen = eval_probs["qwen"].argmax(axis=1)
        pred_majority = majority_vote_pred(eval_probs["bert"], eval_probs["mdeberta"], eval_probs["gemma"], eval_probs["qwen"])
        pred_weighted = weighted_static_pred(eval_probs["bert"], eval_probs["mdeberta"], eval_probs["gemma"], eval_probs["qwen"])
        pred_route = class_conditional_routing_pred(eval_probs["bert"], eval_probs["mdeberta"], eval_probs["gemma"], eval_probs["qwen"])

        append_row(rows, split_key, "Meta_MLP_train_only_12feat", y_true, pred_mlp)
        append_row(rows, split_key, "Meta_LinearSVC_train_only_12feat", y_true, pred_svm)
        append_row(rows, split_key, "Meta_BiLSTM_train_only_12feat", y_true, pred_bilstm)
        append_row(rows, split_key, "Meta_FlatLSTM_train_only_12feat", y_true, pred_flat_lstm)
        append_row(rows, split_key, "BERT", y_true, pred_bert)
        append_row(rows, split_key, "mDeBERTa", y_true, pred_mdeb)
        append_row(rows, split_key, "Gemma", y_true, pred_gemma)
        append_row(rows, split_key, "Qwen", y_true, pred_qwen)
        append_row(rows, split_key, "Majority_vote", y_true, pred_majority)
        append_row(rows, split_key, "Weighted_static_hand_weights", y_true, pred_weighted)
        append_row(rows, split_key, "Class_conditional_routing_computed", y_true, pred_route)
        append_reference_row(rows, split_key)

        plot_confusion(compute_metrics_dict(y_true, pred_mlp)["cm"], f"MLP CM — {split_key}")
        plot_confusion(compute_metrics_dict(y_true, pred_bilstm)["cm"], f"BiLSTM CM — {split_key}")
        plot_confusion(compute_metrics_dict(y_true, pred_flat_lstm)["cm"], f"FlatLSTM CM — {split_key}")

results_df = pd.DataFrame(rows).sort_values(["split", "row_kind", "accuracy"], ascending=[True, True, False], na_position="last")
out_csv = ARTIFACT_DIR / "stacking_results_12feat.csv"
results_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
display(results_df)


## Styled results table

In [ ]:
metric_cols = ["accuracy", "f1_macro", "f1_entailment", "f1_neutral", "f1_contradiction"]
pivot_df = results_df.pivot_table(
    index=["split", "method", "row_kind"], values=metric_cols, aggfunc="first"
).reset_index()
styled = (
    pivot_df.style
    .format({m: "{:.4f}" for m in metric_cols})
    .background_gradient(subset=["accuracy", "f1_macro"], cmap="YlGn")
    .set_properties(**{"text-align": "left"})
    .set_caption("Train-split-only stacking (12 features)")
)
display(styled)


## Feature importance analysis

**Fix applied**: permutation importance is evaluated on **held-out eval data**,
not training data. Using training data systematically underestimates importance
of models the meta-learner memorised less reliably (typically BERT and Gemma),
because permuting their features hurts train-set accuracy less than held-out accuracy.


In [ ]:
def _normalize_importance(vals, names):
    vals = np.array(vals, dtype=np.float64)
    s = vals.sum()
    if s <= 0:
        s = 1.0
    return pd.DataFrame({"feature": names, "importance": vals / s}).sort_values("importance", ascending=False)


def _permutation_importance_custom_blocks(Xs, y, predict_fn, block_defs, n_repeats=5, seed=SEED):
    rng = np.random.default_rng(seed)
    base_acc = accuracy_score(y, predict_fn(Xs))
    names, imps = [], []
    for name, cols in block_defs:
        names.append(name)
        drops = []
        for _ in range(n_repeats):
            Xp = Xs.copy()
            perm = rng.permutation(len(Xp))
            Xp[:, cols] = Xp[perm][:, cols]
            drops.append(base_acc - accuracy_score(y, predict_fn(Xp)))
        imps.append(float(np.mean(drops)))
    return pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False)


block_defs = [
    ("bert_block", [0, 1, 2]),
    ("mdeberta_block", [3, 4, 5]),
    ("gemma_block", [6, 7, 8]),
    ("qwen_block", [9, 10, 11]),
]

importance_rows = []

for cfg in CONFIGS:
    eval_sp = "test_matched" if cfg == "trglue_mnli" else EVAL_SPLITS[cfg][0]
    eb = eval_feature_bank[cfg][eval_sp]
    X_eval_s = eb["X_eval_s"]
    y_eval = eb["y_true"]

    print("\n" + "#"*90)
    print(f"Feature importance for: {cfg}  (eval split: {eval_sp}, n={len(y_eval)})")

    meta = trained_meta[cfg]

    def predict_mlp(X_in):
        with torch.no_grad():
            return meta["mlp"](torch.tensor(X_in, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    def predict_bilstm(X_in):
        with torch.no_grad():
            return meta["bilstm"](torch.tensor(X_in, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    mlp_scores = np.linalg.norm(meta["mlp"].net[0].weight.detach().cpu().numpy(), axis=0)
    print("\nMLP first-layer feature norms (normalized):")
    mlp_norm_df = _normalize_importance(mlp_scores, FEATURE_NAMES)
    display(mlp_norm_df)

    print("MLP permutation importance (eval data):")
    mlp_perm_df = _permutation_importance_custom_blocks(X_eval_s, y_eval, predict_mlp, block_defs)
    display(mlp_perm_df)

    print("BiLSTM permutation importance (eval data):")
    bilstm_perm_df = _permutation_importance_custom_blocks(X_eval_s, y_eval, predict_bilstm, block_defs)
    display(bilstm_perm_df)

    svm_scores = np.linalg.norm(meta["svm"].coef_, axis=0)
    print("LinearSVC coefficient-norm feature importance (normalized):")
    svm_norm_df = _normalize_importance(svm_scores, FEATURE_NAMES)
    display(svm_norm_df)

    for df, method_name in [
        (mlp_norm_df, "MLP_FirstLayerNorm"),
        (mlp_perm_df, "MLP_PermutationImportance_eval"),
        (bilstm_perm_df, "BiLSTM_PermutationImportance_eval"),
        (svm_norm_df, "LinearSVC_CoefNorm"),
    ]:
        d = df.copy()
        d["config"] = cfg
        d["method"] = method_name
        importance_rows.append(d)

all_importances_df = pd.concat(importance_rows, ignore_index=True)
imp_csv = ARTIFACT_DIR / "feature_importances_12feat.csv"
all_importances_df.to_csv(imp_csv, index=False)
print(f"\nFeature importances saved to: {imp_csv}")


## Ablation studies

All ablations target `trglue_mnli::test_matched` (the adversarial benchmark).

**Fixes applied:**
1. MLP ablations (no-Gemma, no-BERT, 8-feat) evaluate on held-out test data.
2. BiLSTM ablations use `build_Xreduced()` which lays out columns as consecutive
   `[p0, p1, p2, entropy]` blocks — safe for `.view(batch, num_models, 4)`.
   The original `Only_Qwen_mDeBERTa` used `[3,4,5, 9,10,11, 13,15]` which
   produced a scrambled sequence when reshaped: that result is now invalid.


In [ ]:
CFG_ABLATION = "trglue_mnli"
SP_ABLATION = "test_matched"

X_train_full = train_feature_bank[CFG_ABLATION]["X_train"]
y_train_abl = train_feature_bank[CFG_ABLATION]["y_train"]
train_probs_abl = train_feature_bank[CFG_ABLATION]["probs"]
eval_probs_abl = eval_feature_bank[CFG_ABLATION][SP_ABLATION]["probs"]
y_true_abl = eval_feature_bank[CFG_ABLATION][SP_ABLATION]["y_true"]

baseline_mlp = results_df.loc[(results_df["split"] == f"{CFG_ABLATION}::{SP_ABLATION}") & (results_df["method"] == "Meta_MLP_train_only_12feat"), "accuracy"].values[0]
baseline_bilstm = results_df.loc[(results_df["split"] == f"{CFG_ABLATION}::{SP_ABLATION}") & (results_df["method"] == "Meta_BiLSTM_train_only_12feat"), "accuracy"].values[0]
baseline_flat_lstm = results_df.loc[(results_df["split"] == f"{CFG_ABLATION}::{SP_ABLATION}") & (results_df["method"] == "Meta_FlatLSTM_train_only_12feat"), "accuracy"].values[0]

print(f"Baselines — MLP: {baseline_mlp:.4f}  BiLSTM: {baseline_bilstm:.4f}  FlatLSTM: {baseline_flat_lstm:.4f}")

ablation_summary = []


def run_mlp_ablation(name, model_keys):
    print(f"\n{'='*70}")
    print(f"MLP ablation: {name}  (models: {model_keys})")

    col_map = {
        "bert": [0, 1, 2],
        "mdeberta": [3, 4, 5],
        "gemma": [6, 7, 8],
        "qwen": [9, 10, 11],
    }
    keep = []
    for k in model_keys:
        keep.extend(col_map[k])
    keep.sort()

    X_tr = X_train_full[:, keep]
    sc = StandardScaler().fit(X_tr)
    Xs_tr = sc.transform(X_tr).astype(np.float32)

    mlp = StackingMLP(in_dim=len(keep), n_classes=NUM_LABELS)
    mlp = _train_torch_model(mlp, Xs_tr, y_train_abl, model_name=f"MLP_{name}")

    X_ev = build_X12(eval_probs_abl)[:, keep]
    Xs_ev = sc.transform(X_ev).astype(np.float32)

    with torch.no_grad():
        Xev_t = torch.tensor(Xs_ev, dtype=torch.float32).to(DEVICE)
        pred = mlp(Xev_t).argmax(-1).cpu().numpy()

    acc = accuracy_score(y_true_abl, pred)
    f1 = f1_score(y_true_abl, pred, average="macro", zero_division=0)

    print(f"MLP {name}: Acc={acc:.4f} | F1={f1:.4f} | Δ vs baseline: {acc-baseline_mlp:+.4f}")

    ablation_summary.append({"learner": "MLP", "config": name, "accuracy": acc, "f1_macro": f1, "delta_vs_baseline": acc - baseline_mlp})


def run_bilstm_ablation(name, model_keys):
    print(f"\n{'='*70}")
    print(f"BiLSTM ablation: {name}  (models: {model_keys})")
    num_models = len(model_keys)

    X_tr = build_Xreduced(train_probs_abl, model_keys)
    sc = StandardScaler().fit(X_tr)
    Xs_tr = sc.transform(X_tr).astype(np.float32)

    bilstm = StackingBiLSTM_dynamic(num_models=num_models)
    bilstm = _train_torch_model(bilstm, Xs_tr, y_train_abl, model_name=f"BiLSTM_{name}")

    X_ev = build_Xreduced(eval_probs_abl, model_keys)
    Xs_ev = sc.transform(X_ev).astype(np.float32)
    with torch.no_grad():
        pred = bilstm(torch.tensor(Xs_ev, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    acc = accuracy_score(y_true_abl, pred)
    f1 = f1_score(y_true_abl, pred, average="macro", zero_division=0)
    print(f"BiLSTM {name}: Acc={acc:.4f} | F1={f1:.4f} | Δ vs BiLSTM baseline: {acc-baseline_bilstm:+.4f}")
    ablation_summary.append({"learner": "BiLSTM", "config": name, "accuracy": acc, "f1_macro": f1, "delta_vs_baseline": acc - baseline_bilstm})


def run_flat_lstm_ablation(name, model_keys):
    print(f"\n{'='*70}")
    print(f"Flat-LSTM ablation: {name}  (models: {model_keys})")

    col_map = {
        "bert": [0, 1, 2],
        "mdeberta": [3, 4, 5],
        "gemma": [6, 7, 8],
        "qwen": [9, 10, 11],
    }
    keep = []
    for k in model_keys:
        keep.extend(col_map[k])
    keep.sort()

    X_tr = X_train_full[:, keep]
    sc = StandardScaler().fit(X_tr)
    Xs_tr = sc.transform(X_tr).astype(np.float32)

    flat_lstm = StackingLSTM_flat(hidden_size=64, n_classes=NUM_LABELS)
    flat_lstm = _train_torch_model(flat_lstm, Xs_tr, y_train_abl, model_name=f"FlatLSTM_{name}")

    X_ev = build_X12(eval_probs_abl)[:, keep]
    Xs_ev = sc.transform(X_ev).astype(np.float32)
    with torch.no_grad():
        pred = flat_lstm(torch.tensor(Xs_ev, dtype=torch.float32).to(DEVICE)).argmax(-1).cpu().numpy()

    acc = accuracy_score(y_true_abl, pred)
    f1 = f1_score(y_true_abl, pred, average="macro", zero_division=0)
    print(f"FlatLSTM {name}: Acc={acc:.4f} | F1={f1:.4f} | Δ vs FlatLSTM baseline: {acc-baseline_flat_lstm:+.4f}")
    ablation_summary.append({"learner": "FlatLSTM", "config": name, "accuracy": acc, "f1_macro": f1, "delta_vs_baseline": acc - baseline_flat_lstm})


### MLP ablations


In [ ]:
run_mlp_ablation("No_Gemma",             ["bert", "mdeberta", "qwen"])
run_mlp_ablation("No_BERT",              ["mdeberta", "gemma", "qwen"])
run_mlp_ablation("Only_Qwen_mDeBERTa",  ["mdeberta", "qwen"])


### BiLSTM ablations (fixed column ordering)

In [ ]:
run_bilstm_ablation("No_Gemma",            ["bert", "mdeberta", "qwen"])
run_bilstm_ablation("No_BERT",             ["mdeberta", "gemma", "qwen"])
run_bilstm_ablation("Only_Qwen_mDeBERTa", ["mdeberta", "qwen"])


### Flat LSTM ablations (StackingLSTM_flat)

In [ ]:
run_flat_lstm_ablation("Full",               ["bert", "mdeberta", "gemma", "qwen"])
run_flat_lstm_ablation("No_Gemma",           ["bert", "mdeberta", "qwen"])
run_flat_lstm_ablation("No_BERT",            ["mdeberta", "gemma", "qwen"])
run_flat_lstm_ablation("Only_Qwen_mDeBERTa", ["mdeberta", "qwen"])

### Per-model normalization ablation (MLP)


In [ ]:
print("\n" + "="*70)
print("MLP ablation: Per-model normalization")

def per_model_normalize_train(X_train):
    groups = {
        "bert": [0, 1, 2],
        "mdeberta": [3, 4, 5],
        "gemma": [6, 7, 8],
        "qwen": [9, 10, 11],
    }
    scalers = {}
    X_out = X_train.copy().astype(np.float32)
    for k, cols in groups.items():
        sc = StandardScaler().fit(X_train[:, cols])
        X_out[:, cols] = sc.transform(X_train[:, cols])
        scalers[k] = (sc, cols)
    return X_out, scalers

def per_model_normalize_apply(X, scalers):
    X_out = X.copy().astype(np.float32)
    for _, (sc, cols) in scalers.items():
        X_out[:, cols] = sc.transform(X[:, cols])
    return X_out

X_tr_pm, pm_scalers = per_model_normalize_train(X_train_full)
sc_global = StandardScaler().fit(X_tr_pm)
Xs_tr_pm = sc_global.transform(X_tr_pm).astype(np.float32)

mlp_pm = StackingMLP(in_dim=12, n_classes=NUM_LABELS)
mlp_pm = _train_torch_model(mlp_pm, Xs_tr_pm, y_train_abl, model_name="MLP_per_model_norm")

X_ev_pm = per_model_normalize_apply(build_X12(eval_probs_abl), pm_scalers)
Xs_ev_pm = sc_global.transform(X_ev_pm).astype(np.float32)
with torch.no_grad():
    Xev_t = torch.tensor(Xs_ev_pm, dtype=torch.float32).to(DEVICE)
    pred_pm = mlp_pm(Xev_t).argmax(-1).cpu().numpy()

acc_pm = accuracy_score(y_true_abl, pred_pm)
f1_pm = f1_score(y_true_abl, pred_pm, average="macro", zero_division=0)

print(f"MLP per-model-norm: Acc={acc_pm:.4f} | F1={f1_pm:.4f} | Δ: {acc_pm-baseline_mlp:+.4f}")

ablation_summary.append({"learner": "MLP", "config": "Per_model_norm", "accuracy": acc_pm, "f1_macro": f1_pm, "delta_vs_baseline": acc_pm - baseline_mlp})


### BiLSTM with per-model normalization ablation

In [ ]:
print("\n" + "="*70)
print("BiLSTM ablation: Per-model normalization")

# Reuse the per_model_normalize helpers defined in the MLP per-model-norm cell.
X_tr_pm_bi, pm_scalers_bi = per_model_normalize_train(X_train_full)
sc_global_bi = StandardScaler().fit(X_tr_pm_bi)
Xs_tr_pm_bi  = sc_global_bi.transform(X_tr_pm_bi).astype(np.float32)

bilstm_pm = StackingBiLSTM(hidden_size=64, n_classes=NUM_LABELS)
bilstm_pm = _train_torch_model(bilstm_pm, Xs_tr_pm_bi, y_train_abl, model_name="BiLSTM_per_model_norm")

X_ev_pm_bi  = per_model_normalize_apply(build_X12(eval_probs_abl), pm_scalers_bi)
Xs_ev_pm_bi = sc_global_bi.transform(X_ev_pm_bi).astype(np.float32)
with torch.no_grad():
    pred_pm_bi = bilstm_pm(
        torch.tensor(Xs_ev_pm_bi, dtype=torch.float32).to(DEVICE)
    ).argmax(-1).cpu().numpy()

acc_pm_bi = accuracy_score(y_true_abl, pred_pm_bi)
f1_pm_bi  = f1_score(y_true_abl, pred_pm_bi, average="macro", zero_division=0)
print(f"BiLSTM per-model-norm: Acc={acc_pm_bi:.4f} | F1={f1_pm_bi:.4f} | Δ: {acc_pm_bi-baseline_bilstm:+.4f}")
ablation_summary.append({
    "learner": "BiLSTM", "config": "Per_model_norm", "accuracy": acc_pm_bi,
    "f1_macro": f1_pm_bi, "delta_vs_baseline": acc_pm_bi - baseline_bilstm,
})

### Ablation summary table

In [ ]:
abl_df = pd.DataFrame(ablation_summary)
abl_df["baseline_acc"] = abl_df["learner"].map({
    "MLP": baseline_mlp,
    "BiLSTM": baseline_bilstm,
    "FlatLSTM": baseline_flat_lstm,
})
abl_df = abl_df.sort_values(["learner", "accuracy"], ascending=[True, False])
display(abl_df.style
    .format({"accuracy": "{:.4f}", "f1_macro": "{:.4f}",
             "delta_vs_baseline": "{:+.4f}", "baseline_acc": "{:.4f}"})
    .background_gradient(subset=["accuracy"], cmap="YlGn")
    .set_caption(f"Ablation results on {CFG_ABLATION}::{SP_ABLATION} "
                 f"(MLP / BiLSTM / FlatLSTM)"))

abl_csv = ARTIFACT_DIR / "ablation_summary_12feat.csv"
abl_df.to_csv(abl_csv, index=False)
print("Saved:", abl_csv)


## Save trained meta-learners

In [ ]:
import shutil

models_dir = ARTIFACT_DIR / "trained_models"
models_dir.mkdir(exist_ok=True, parents=True)

for cfg, meta in trained_meta.items():
    joblib.dump(meta["scaler"], models_dir / f"{cfg}_scaler.joblib")
    torch.save(meta["mlp"].state_dict(), models_dir / f"{cfg}_mlp.pt")
    joblib.dump(meta["svm"], models_dir / f"{cfg}_svm.joblib")
    torch.save(meta["bilstm"].state_dict(), models_dir / f"{cfg}_bilstm.pt")
    torch.save(meta["flat_lstm"].state_dict(), models_dir / f"{cfg}_flat_lstm.pt")
    print(f"Saved meta-learners for {cfg}")

shutil.make_archive("stacking_artifacts_12feat", "zip", ARTIFACT_DIR)
print("Zipped stacking_artifacts_12feat.zip")


## Flat LSTM vs Grouped BiLSTM (12 Features)

With the 12-feature setup, each model contributes only `[p0, p1, p2]`.

- **Grouped BiLSTM** reshapes `(batch, 12)` to `(batch, 4, 3)`, so each timestep corresponds to one model.
- **Flat LSTM** reshapes `(batch, 12)` to `(batch, 12, 1)`, so each timestep is an individual scalar feature.

The grouped variant typically remains easier to optimize because per-model probability triplets are kept together, while the flat variant must reconstruct that structure from scalar sequence order.


## Upload Files

In [ ]:
from google.colab import files

# Upload files from your local system
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


## Unzip File

In [ ]:
# Replace 'your_zip_file.zip' with the actual name of your uploaded zip file.
# The files will be extracted to the current working directory (/content/).
!unzip -q /content/stacking_cache_train_split_12feat.zip -d /content/

print("To unzip your file, uncomment the line above and replace 'your_zip_file.zip' with the actual filename.")
print("For example: !unzip -q /content/stacking_cache_train_split_12feat.zip -d /content/")


## Summary for this 12-feature version

### Key updates applied

- Switched cache input to `stacking_cache_train_split_12feat`.
- Replaced 16-feature builders with 12-feature builders (`build_X12`, 3 probs per model).
- Uses the original notebook-family MLP only (`StackingMLP`).
- Main evaluation reports `Meta_MLP_train_only_12feat` alongside SVC, BiLSTM, and FlatLSTM.
- Feature-importance and ablation studies are aligned with the single-MLP setup.
- Artifact exports are saved under `stacking_artifacts_12feat`.
